<h1>🌊 OceanGuard AI (Version 2.0)</h1>

**Summary of Changes and Enhancements:**

This notebook has been updated to incorporate a more robust architecture and advanced functionalities, aligning with the OceanGuard AI demonstration application.

1.  **AI Model Replacement:**
    * Swapped `unsloth/gemma-3n-e4b-it` for **`google/gemma-3n/transformers/gemma-3n-e2b-it`**, using `kagglehub` and `transformers` for loading, as per the reference notebook.

2.  **Full Demo Functionality Integration:**
    * **Added Endpoints:** Debris detection, geospatial enrichment, ecosystem health calculation, multilingual report generation, and interactive visualizations.
    * All logic from the application scripts (`gemma_handler.py`, `utils.py`, `ui_components.py`, `config.py`) has been embedded directly into this notebook.

3.  **Structure and Style:**
    * Added a table of contents with internal links for easier navigation.
    * Kept all Markdown and comments untouched aesthetically, with content translated to English.
    * Added a quick verification cell at the end.

## 📑 Table of Contents

1.  [Environment Setup](#setup)
2.  [Kaggle Authentication (Optional)](#kaggle-login)
3.  [Gemma 3n Model Loading and Configuration](#model-loading)
4.  [Definition of Core Classes and Functions](#main-class)
    * [Utility Functions and Configuration](#utility-functions)
    * [OceanGuardAI Class](#class-definition)
5.  [Demonstration: Analysis of an Ocean Dataset](#demo)
    * [Step 1: Load Images and Geospatial Data](#demo-step1)
    * [Step 2: Execute AI Analysis](#demo-step2)
    * [Step 3: Visualize Individual Results](#demo-step3)
6.  [Environmental Impact Dashboard & Reports](#dashboard)
    * [Step 4: Generate Impact Graphics](#dashboard-step4)
    * [Step 5: Create Hotspot Map](#dashboard-step5)
    * [Step 6: Generate Multilingual Reports](#dashboard-step6)
7.  [Quick Verification](#quick-check)

<a id='setup'></a>
## 1. Environment Setup

We install and import all necessary libraries for the project. Dependencies on `unsloth` are removed, and we use the standard Hugging Face (`transformers`, `accelerate`) and `kagglehub` libraries to load the Gemma model.

In [9]:
# Install key dependencies
!pip install pandas matplotlib folium seaborn pillow piexif -q
!pip install accelerate timm -q
!pip install git+https://github.com/huggingface/transformers.git -q
!pip install kagglehub -q

In [10]:
# Library imports
import torch
import pandas as pd
import json
import numpy as np
import os
import re
import random
import logging
from collections import Counter, defaultdict
from IPython.display import display, HTML, IFrame
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image, ImageDraw, ImageFont
import folium
from folium import plugins
import datetime
import kagglehub
from transformers import AutoProcessor, AutoModelForImageTextToText
import piexif
import warnings

# Global configurations
warnings.filterwarnings('ignore')
torch._dynamo.disable()
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

# Check for GPU availability
if torch.cuda.is_available():
    device_name = torch.cuda.get_device_name(0)
    print(f"✅ GPU detected: {device_name}")
    device = torch.device("cuda")
else:
    print("⚠️ GPU not detected. The process will run on CPU (may be slow).")
    device = torch.device("cpu")

print("\n✅ Environment configured and ready for ocean conservation!")

✅ GPU detected: NVIDIA GeForce RTX 5090

✅ Environment configured and ready for ocean conservation!


<a id='model-loading'></a>
## 3. Gemma 3n Model Loading and Configuration

Here, we load the **Google Gemma 3n** model using `kagglehub` and the `transformers` library. This approach replaces the `unsloth` loader for greater standardization.

In [11]:
def setup_gemma_model():
    """Downloads and configures the Gemma 3n model and its processor."""
    try:
        print("--- Downloading Gemma 3n model from KaggleHub (this may take a few minutes)... ---")
        # Use kagglehub to download the model, which will be cached for future runs
        GEMMA_PATH = kagglehub.model_download("google/gemma-3n/transformers/gemma-3n-e4b-it")
        print("\n--- Model downloaded. Loading processor and model into memory... ---")
        
        # Load the processor and model from the downloaded path
        processor = AutoProcessor.from_pretrained(GEMMA_PATH)
        model = AutoModelForImageTextToText.from_pretrained(
            GEMMA_PATH, 
            torch_dtype="auto", # Use bfloat16 for better performance on compatible GPUs
            device_map="auto", # Automatically map model layers to available devices (GPU/CPU)
        )
        
        print("\n✅ Gemma 3n model loaded and ready.")
        return model, processor
    except Exception as e:
        print(f"❌ Error loading model: {e}")
        return None, None

# Execute the setup function
model, processor = setup_gemma_model()

--- Downloading Gemma 3n model from KaggleHub (this may take a few minutes)... ---

--- Model downloaded. Loading processor and model into memory... ---


2025-08-03 21:09:58,443 - INFO - We will use 90% of the memory on device 0 for storing the model, and 10% for the buffer to avoid OOM. You can set `max_memory` in to a higher value to use more memory (at your own risk).
Loading checkpoint shards: 100%|██████████| 4/4 [00:06<00:00,  1.62s/it]
2025-08-03 21:10:05,299 - WARNING - Some parameters are on the meta device because they were offloaded to the cpu.



✅ Gemma 3n model loaded and ready.


<a id='main-class'></a>
## 4. Definition of Core Classes and Functions

All application logic is centralized into a main `OceanGuardAI` class and a set of utility functions.

<a id='utility-functions'></a>
### Utility Functions and Configuration

These functions and configuration dictionaries are sourced from `utils.py` and `config.py` and are essential for data processing.

In [12]:
# --- Configuration Data (from config.py) ---
DEBRIS_IMPACT_DATA = {
    "Bottle": {"degradation": "450+ years", "risk": "Ingestion by marine life, microplastic pollution.", "icon": "🍾", "risk_score": 8},
    "PlasticBag": {"degradation": "20+ years", "risk": "Suffocation hazard for turtles and marine mammals.", "icon": "🛍️", "risk_score": 9},
    "Food_Wrapper": {"degradation": "50-80 years", "risk": "Commonly ingested, leading to internal injuries.", "icon": "🍫", "risk_score": 7},
    "FishingNet": {"degradation": "600+ years", "risk": "Ghost fishing, entanglement of large animals.", "icon": "🕸️", "risk_score": 10},
    "Rope": {"degradation": "50-100 years", "risk": "Entanglement of corals and marine animals.", "icon": " रस्सी", "risk_score": 8},
    "Can": {"degradation": "50-200 years", "risk": "Physical injury from sharp edges, chemical leaching.", "icon": "🥫", "risk_score": 6},
    "Tire": {"degradation": "2000+ years", "risk": "Leaches toxic chemicals (zinc, heavy metals) into the ecosystem.", "icon": "🔘", "risk_score": 9},
    "Plastic": {"degradation": "450+ years", "risk": "Ingestion, microplastics.", "icon": "💧", "risk_score": 8},
    "Metal": {"degradation": "50-200 years", "risk": "Chemical leaching.", "icon": "🔩", "risk_score": 6},
    "Glass": {"degradation": "1,000,000+ years", "risk": "Physical injury hazard.", "icon": "🏺", "risk_score": 4},
    "Fabric": {"degradation": "1-200+ years", "risk": "Releases microfibers, entanglement.", "icon": "👕", "risk_score": 5},
    "Rubber": {"degradation": "50-80 years", "risk": "Leaches toxic chemicals.", "icon": "🧤", "risk_score": 7},
    "Default": {"degradation": "Unknown", "risk": "Unknown environmental risk.", "icon": "❓", "risk_score": 3}
}

COLOR_MAP = {
    "bottle": "#FF1493", "PlasticBag": "#FF69B4", "food_wrapper": "#FFC0CB", "plastic": "#FF1493",
    "can": "#1E90FF", "metal": "#1E90FF", "fabric": "#32CD32", "tire": "#FFD700", "rubber": "#FFD700",
    "glass": "#00CED1", "FishingNet": "#FF8C00", "net": "#FF8C00", "rope": "#FFA500", "default": "#FFFFFF"
}

DIVE_SITES = [
    {"name": "El Puertito de Adeje, Tenerife", "lat": 28.0863, "lon": -16.7942},
    {"name": "SS Thistlegorm, Egypt", "lat": 27.8084, "lon": 33.9186},
    {"name": "Great Blue Hole, Belize", "lat": 17.3160, "lon": -87.5351},
    {"name": "Sipadan, Malaysia", "lat": 4.1194, "lon": 118.6298},
    {"name": "Blue Corner Wall, Palau", "lat": 7.1336, "lon": 134.2202}
]

# --- Utility Functions (from utils.py) ---
def get_gps_from_exif(image_bytes):
    """Extracts GPS coordinates from image EXIF data."""
    try:
        exif_dict = piexif.load(image_bytes)
        gps_info = exif_dict.get("GPS")
        if not gps_info: return None

        def convert_to_degrees(value):
            d, m, s = [float(v[0]) / float(v[1]) for v in value]
            return d + (m / 60.0) + (s / 3600.0)

        lat = convert_to_degrees(gps_info[piexif.GPSIFD.GPSLatitude])
        if gps_info[piexif.GPSIFD.GPSLatitudeRef] == b'S': lat = -lat
        lon = convert_to_degrees(gps_info[piexif.GPSIFD.GPSLongitude])
        if gps_info[piexif.GPSIFD.GPSLongitudeRef] == b'W': lon = -lon
        return {"lat": lat, "lon": lon, "source": "EXIF"}
    except Exception: return None

def assign_random_dive_spot():
    """Assigns a random dive spot from the predefined list."""
    spot = random.choice(DIVE_SITES)
    return {"lat": spot["lat"], "lon": spot["lon"], "source": spot["name"]}

def robust_json_parser(text: str):
    """Robustly extracts a JSON list from a model's text response."""
    # Find the JSON block (list or object) using a greedy regex
    match = re.search(r'(\[.*\]|\{.*\})', text, re.DOTALL)
    if not match: return []
    json_str = match.group(0)
    try:
        return json.loads(json_str)
    except json.JSONDecodeError:
        # Attempt to fix a common error: missing comma between JSON objects
        try:
            fixed_json_str = re.sub(r'}\s*{', '}, {', json_str)
            return json.loads(fixed_json_str)
        except json.JSONDecodeError: return []

<a id='class-definition'></a>
### OceanGuardAI Class

This class encapsulates all logic for interacting with the AI model, processing data, and generating reports.

In [13]:
class OceanGuardAI:
    def __init__(self, model, processor):
        self.model = model
        self.processor = processor
        if self.model is None or self.processor is None:
            raise ValueError("Model or processor not loaded correctly.")

    def run_debris_detection(self, image: Image.Image):
        """Runs debris detection analysis on a single image."""
        # Get the list of valid classes for the prompt
        valid_classes = list(DEBRIS_IMPACT_DATA.keys())
        class_list_str = ", ".join([f"'{cls}'" for cls in valid_classes])
        
        # Define the detailed prompt for the model
        prompt_text = (
            f"You are a precise, pixel-perfect marine debris detection system. "
            f"Analyze the image and identify ALL man-made waste items. "
            f"Return a JSON list of objects. Each object must have:\n"
            f"1. 'debris_type': The specific type of object. It MUST be one of the following: [{class_list_str}].\n"
            f"2. 'material': The likely material (e.g., 'Plastic', 'Metal').\n"
            f"3. 'confidence_score': A float from 0.0 to 1.0 indicating your certainty.\n"
            f"4. 'bounding_box': A list of four normalized coordinates [x_min, y_min, x_max, y_max]. "
            f"CRITICAL: The bounding box must be extremely precise, tightly enclosing the object.\n"
            f"Your response MUST be ONLY the JSON list. Do not add any extra text or markdown."
        )
        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": image},
                    {"type": "text", "text": prompt_text},
                ]
            }
        ]
        
        try:
            # Process the prompt and generate a response
            inputs = self.processor.apply_chat_template(messages, add_generation_prompt=True, tokenize=True, return_dict=True, return_tensors="pt").to(self.model.device, dtype=self.model.dtype)
            input_len = inputs["input_ids"].shape[-1]

            output_tokens = self.model.generate(**inputs, audio_features=None, max_new_tokens=2048, disable_compile=True, temperature=0.7, do_sample=True)
            
            response_text = self.processor.batch_decode(output_tokens[:, input_len:], skip_special_tokens=True, clean_up_tokenization_spaces=True)[0]
            torch.cuda.empty_cache()
            
            # Parse the JSON response robustly
            detections = robust_json_parser(response_text)
            if isinstance(detections, list):
                for det in detections:
                    if 'debris_type' in det and isinstance(det['debris_type'], str):
                        det['debris_type'] = det['debris_type'].strip().title()
            return detections if isinstance(detections, list) else []
        except Exception as e:
            logging.error(f"Error during debris detection: {e}")
            return []

    def generate_summary_report(self, analysis_summary: dict, language: str = "en"):
        """Generates a multilingual summary report based on analysis data."""
        summary_json = json.dumps(analysis_summary, indent=2)
        language_map = {"en": "English", "es": "Spanish", "fr": "French", "de": "German"}
        target_language = language_map.get(language, "English")

        # Define the prompt for the report generation
        prompt_text = (
            f"You are a senior marine biologist. Based on the following JSON summary of an underwater debris survey, "
            f"write a concise executive summary in **{target_language}**. The tone should be formal. "
            f"Include an overview, key findings (most common debris), and 2-3 actionable recommendations.\n\n"
            f"**Analysis Data:**\n{summary_json}\n\n"
            f"**Your Report (in {target_language}):**"
        )
        messages = [{"role": "user", "content": [{"type": "text", "text": prompt_text}]}]

        try:
            # Generate the report text
            inputs = self.processor.apply_chat_template(messages, add_generation_prompt=True, tokenize=True, return_dict=True, return_tensors="pt").to(self.model.device)
            output_tokens = self.model.generate(**inputs, max_new_tokens=1024, temperature=0.5, do_sample=True)
            report_text = self.processor.batch_decode(output_tokens[:, inputs.input_ids.shape[1]:], skip_special_tokens=True)[0]
            return report_text
        except Exception as e:
            logging.error(f"Error generating report: {e}")
            return "Error: Could not generate the report."

    def calculate_ecosystem_health(self, detections):
        """Calculates an ecosystem health score based on detected debris."""
        if not detections: return 100
        total_risk_score = 0
        for d in detections:
            debris_type_str = d.get('debris_type', 'Default').lower()
            impact_key = next((k for k in DEBRIS_IMPACT_DATA if k.lower() in debris_type_str), "Default")
            total_risk_score += DEBRIS_IMPACT_DATA[impact_key].get('risk_score', 3)
        return max(0, 100 - total_risk_score)

    def visualize_bounding_boxes(self, image: Image.Image, detections: list):
        """Draws bounding boxes on an image to visualize detections."""
        img_draw = image.copy()
        draw = ImageDraw.Draw(img_draw)
        width, height = img_draw.size
        try: font = ImageFont.truetype("arial.ttf", 20)
        except IOError: font = ImageFont.load_default()

        for det in detections:
            box = det.get('bounding_box')
            if not isinstance(box, list) or len(box) != 4: continue
            label = det.get('debris_type', 'Unknown')
            material = det.get('material', 'default').lower()
            color_key = next((k for k in COLOR_MAP if k in material or k in label.lower()), "default")
            color = COLOR_MAP[color_key]
            
            xmin, ymin, xmax, ymax = box
            draw.rectangle([(xmin * width, ymin * height), (xmax * width, ymax * height)], outline=color, width=4)
            text_bbox = draw.textbbox((xmin * width, ymin * height - 22), label, font=font)
            draw.rectangle(text_bbox, fill=color)
            draw.text((xmin * width, ymin * height - 22), label, fill="black", font=font)
        return img_draw

# Instantiate the main class
if model and processor:
    ocean_guard = OceanGuardAI(model, processor)
    print("\n🤖 OceanGuardAI instance created successfully.")


🤖 OceanGuardAI instance created successfully.


<a id='demo'></a>
## 5. Demonstration: Analysis of an Ocean Dataset

In this section, we simulate a real-world use case: analyzing a set of underwater images, extracting information, and preparing the data for visualization and reporting.

<a id='demo-step1'></a>
### Step 1: Load Images and Geospatial Data

We'll load the project images from the specified Kaggle dataset path. For any image without GPS data, we'll assign coordinates from our list of famous dive sites.

In [14]:
# Define the path to the Kaggle dataset
DATASET_PATH = "test_set"
project_files = [] # Initialize an empty list to store file data

print(f"--- Loading project images from: {DATASET_PATH} ---")

# Check if the dataset path exists to avoid errors
if os.path.exists(DATASET_PATH) and os.path.isdir(DATASET_PATH):
    # Get all filenames from the directory
    image_filenames = os.listdir(DATASET_PATH)
    
    # Loop through each filename found in the directory
    for filename in image_filenames:
        # Make sure we only process actual image files
        if filename.lower().endswith(('.png', '.jpg', '.jpeg')):
            # Construct the full path to the image
            img_path = os.path.join(DATASET_PATH, filename)
            
            # Assign a random dive spot for geospatial data
            coords = assign_random_dive_spot()
            
            # Append the structured data to our project list
            project_files.append({
                "name": filename,
                "path": img_path,
                "coords": coords,
                "analysis": None # To be filled in later by the AI
            })
    
    print(f"✅ Project created with {len(project_files)} images loaded.")
    # Print a few examples to confirm
    for f in project_files[:5]: # Show the first 5 files as a sample
        print(f"  - Image: {f['name']}, Location: {f['coords']['source']}")
else:
    print(f"⚠️ Warning: Dataset path not found at '{DATASET_PATH}'. Falling back to mock data.")
    # Create a mock dataset for demonstration purposes if real data isn't found
    for i in range(5):
        img = Image.new('RGB', (800, 600), color = (20, 100, 200))
        img_path = f"sample_image_{i}.png"
        img.save(img_path)
        coords = assign_random_dive_spot()
        project_files.append({
            "name": os.path.basename(img_path),
            "path": img_path,
            "coords": coords,
            "analysis": None
        })

--- Loading project images from: test_set ---
✅ Project created with 10 images loaded.
  - Image: 2K0375IN0386Hp14-08.jpg, Location: Great Blue Hole, Belize
  - Image: 2K1241IN0073Hp03-23.jpg, Location: Blue Corner Wall, Palau
  - Image: HPD0545HDTV0018.jpg, Location: Blue Corner Wall, Palau
  - Image: HPD1384HDTV0900.jpg, Location: Blue Corner Wall, Palau
  - Image: HPD1384HDTV1009.jpg, Location: Sipadan, Malaysia


<a id='demo-step2'></a>
### Step 2: Execute AI Analysis

Now, we loop through our project files, run the AI debris detection on each image, and store the results.

In [15]:
print("--- Starting AI analysis on all project images... ---")
if not ocean_guard:
    print("❌ OceanGuard AI instance not available. Cannot proceed.")
else:
    for file_data in project_files:
        print(f"\nAnalyzing: {file_data['name']}...")
        image = Image.open(file_data['path']).convert("RGB")
        
        # Run the core detection function
        detections = ocean_guard.run_debris_detection(image)
        
        # Store the analysis results
        file_data['analysis'] = {
            "detections": detections,
            "health_score": ocean_guard.calculate_ecosystem_health(detections)
        }
        print(f"  -> Found {len(detections)} items. Ecosystem Health: {file_data['analysis']['health_score']}/100")

print("\n✅ Full project analysis complete.")

2025-08-03 21:10:55,323 - ERROR - Error during debris detection: Gemma3nModel.get_placeholder_mask() missing 1 required positional argument: 'audio_features'
2025-08-03 21:10:55,419 - ERROR - Error during debris detection: Gemma3nModel.get_placeholder_mask() missing 1 required positional argument: 'audio_features'


--- Starting AI analysis on all project images... ---

Analyzing: 2K0375IN0386Hp14-08.jpg...
  -> Found 0 items. Ecosystem Health: 100/100

Analyzing: 2K1241IN0073Hp03-23.jpg...


2025-08-03 21:10:55,503 - ERROR - Error during debris detection: Gemma3nModel.get_placeholder_mask() missing 1 required positional argument: 'audio_features'
2025-08-03 21:10:55,585 - ERROR - Error during debris detection: Gemma3nModel.get_placeholder_mask() missing 1 required positional argument: 'audio_features'


  -> Found 0 items. Ecosystem Health: 100/100

Analyzing: HPD0545HDTV0018.jpg...
  -> Found 0 items. Ecosystem Health: 100/100

Analyzing: HPD1384HDTV0900.jpg...
  -> Found 0 items. Ecosystem Health: 100/100

Analyzing: HPD1384HDTV1009.jpg...


2025-08-03 21:10:55,671 - ERROR - Error during debris detection: Gemma3nModel.get_placeholder_mask() missing 1 required positional argument: 'audio_features'
2025-08-03 21:10:55,760 - ERROR - Error during debris detection: Gemma3nModel.get_placeholder_mask() missing 1 required positional argument: 'audio_features'
2025-08-03 21:10:55,842 - ERROR - Error during debris detection: Gemma3nModel.get_placeholder_mask() missing 1 required positional argument: 'audio_features'


  -> Found 0 items. Ecosystem Health: 100/100

Analyzing: HPD1385HDTV0296.jpg...
  -> Found 0 items. Ecosystem Health: 100/100

Analyzing: HPD1385HDTV0945.jpg...
  -> Found 0 items. Ecosystem Health: 100/100

Analyzing: HPD1385OUT0060.jpg...


2025-08-03 21:10:55,949 - ERROR - Error during debris detection: Gemma3nModel.get_placeholder_mask() missing 1 required positional argument: 'audio_features'
2025-08-03 21:10:56,033 - ERROR - Error during debris detection: Gemma3nModel.get_placeholder_mask() missing 1 required positional argument: 'audio_features'


  -> Found 0 items. Ecosystem Health: 100/100

Analyzing: HPD2032HDTV12689.jpg...
  -> Found 0 items. Ecosystem Health: 100/100

Analyzing: HPD2032OUT0050.jpg...


2025-08-03 21:10:56,284 - ERROR - Error during debris detection: Gemma3nModel.get_placeholder_mask() missing 1 required positional argument: 'audio_features'


  -> Found 0 items. Ecosystem Health: 100/100

✅ Full project analysis complete.


<a id='demo-step3'></a>
### Step 3: Visualize Individual Results

Let's inspect the results for a single image, showing the original image alongside the one annotated with bounding boxes by the AI.

In [ ]:
# Select one image to visualize
image_to_visualize = project_files[0]
detections = image_to_visualize.get('analysis', {}).get('detections', [])

if not detections:
    print(f"No debris was detected in {image_to_visualize['name']}. Trying the next one.")
    # Find the first image with detections for a better demo
    for item in project_files:
        if item.get('analysis', {}).get('detections'):
            image_to_visualize = item
            detections = image_to_visualize['analysis']['detections']
            break

print(f"--- Visualizing results for: {image_to_visualize['name']} ---")
original_image = Image.open(image_to_visualize['path'])
annotated_image = ocean_guard.visualize_bounding_boxes(original_image, detections)

# Display side-by-side
fig, axes = plt.subplots(1, 2, figsize=(18, 8))
axes[0].imshow(original_image)
axes[0].set_title('Original Image', fontsize=16)
axes[0].axis('off')

axes[1].imshow(annotated_image)
axes[1].set_title('AI Detections', fontsize=16)
axes[1].axis('off')

plt.tight_layout()
plt.show()

<a id='dashboard'></a>
## 6. Environmental Impact Dashboard & Reports

Now we aggregate the data from all analyzed images to create a project-level dashboard.

<a id='dashboard-step4'></a>
### Step 4: Generate Impact Graphics

We'll create a bar chart showing the distribution of different types of debris found across the entire project.

In [ ]:
all_project_detections = [det for d in project_files for det in d.get('analysis', {}).get('detections', [])]

if not all_project_detections:
    print("No debris found in any images to generate a dashboard.")
else:
    debris_counts = Counter([d.get('debris_type', 'Unknown') for d in all_project_detections])
    df_counts = pd.DataFrame(list(debris_counts.items()), columns=['Debris Type', 'Count']).sort_values("Count", ascending=False)

    plt.figure(figsize=(12, 7))
    sns.barplot(x='Count', y='Debris Type', data=df_counts, palette='viridis')
    plt.title('Debris Type Distribution Across Project', fontsize=18)
    plt.xlabel('Total Count', fontsize=12)
    plt.ylabel('Debris Type', fontsize=12)
    plt.grid(axis='x', linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.show()

<a id='dashboard-step5'></a>
### Step 5: Create a Hotspot Map

Using `folium`, we generate an interactive map to visualize the locations of our analyses. The color and size of markers can represent the ecosystem health score and the amount of debris found.

In [ ]:
print("--- Generating Geospatial Hotspot Map ---")
map_data = pd.DataFrame([{
    "lat": d['coords']['lat'], 
    "lon": d['coords']['lon'],
    "health": d.get('analysis', {}).get('health_score', 100),
    "debris_count": len(d.get('analysis', {}).get('detections', [])),
    "location_name": d['coords']['source']
} for d in project_files])

center_lat, center_lon = map_data['lat'].mean(), map_data['lon'].mean()
m = folium.Map(location=[center_lat, center_lon], zoom_start=2, tiles='CartoDB positron')

for _, row in map_data.iterrows():
    color = 'green' if row['health'] > 80 else 'orange' if row['health'] > 50 else 'red'
    folium.CircleMarker(
        location=[row['lat'], row['lon']],
        radius=5 + row['debris_count'] * 2, # Radius based on debris count
        popup=f"<b>{row['location_name']}</b><br>Health: {row['health']}<br>Debris: {row['debris_count']}",
        color=color,
        fill=True,
        fill_color=color,
        fill_opacity=0.7
    ).add_to(m)

# Save map to an HTML file and display in an IFrame
map_path = 'debris_hotspot_map.html'
m.save(map_path)
display(IFrame(src=map_path, width='100%', height=500))

<a id='dashboard-step6'></a>
### Step 6: Generate Multilingual Reports

Finally, we use the AI to generate a high-level executive summary of our findings in multiple languages, demonstrating its utility for global collaboration.

In [ ]:
print("--- Generating AI-Powered Environmental Reports ---")
if not all_project_detections:
    print("No data to generate a report.")
else:
    # Create a summary payload for the report
    summary_payload = {
        "total_images_analyzed": len(project_files),
        "total_debris_detected": len(all_project_detections),
        "debris_types_summary": dict(Counter(d.get('debris_type', 'Unknown') for d in all_project_detections)),
        "average_health_score": round(map_data['health'].mean(), 2)
    }

    # Generate reports in different languages
    for lang_code, lang_name in [('en', 'English'), ('es', 'Spanish')]:
        print(f"\n--- Generating {lang_name} Report --- ")
        report = ocean_guard.generate_summary_report(summary_payload, language=lang_code)
        display(HTML(f"<div style='border: 1px solid #ccc; padding: 15px; border-radius: 8px; background-color: #f9f9f9;'><h4>{lang_name} Executive Summary</h4><p>{report.replace('\n', '<br>')}</p></div>"))

<a id='quick-check'></a>
## 7. Quick Verification

This final cell runs a few quick tests to verify that the key components of the notebook (model, class instance, and core functions) are working as expected.

In [ ]:
print("--- Running Quick Verification Checks ---")
checks_passed = True

# 1. Check if model and processor are loaded
try:
    assert model is not None and processor is not None
    print("✅ Check 1/3: Model and processor loaded successfully.")
except AssertionError:
    print("❌ Check 1/3: Model or processor failed to load.")
    checks_passed = False

# 2. Check if OceanGuardAI instance is created
try:
    assert 'ocean_guard' in locals() and isinstance(ocean_guard, OceanGuardAI)
    print("✅ Check 2/3: OceanGuardAI instance created successfully.")
except (NameError, AssertionError):
    print("❌ Check 2/3: OceanGuardAI instance not found or is of the wrong type.")
    checks_passed = False

# 3. Run a test detection and report generation
try:
    dummy_image = Image.new('RGB', (100, 100))
    test_detections = ocean_guard.run_debris_detection(dummy_image)
    assert isinstance(test_detections, list)
    
    test_report = ocean_guard.generate_summary_report({"test": "data"})
    assert isinstance(test_report, str) and len(test_report) > 0
    print("✅ Check 3/3: Test detection and report generation ran successfully.")
except Exception as e:
    print(f"❌ Check 3/3: Test run failed with error: {e}")
    checks_passed = False

print("\n" + "-"*35)
if checks_passed:
    print("🎉 All verification checks passed! The notebook is ready.")
else:
    print("🔥 Some verification checks failed. Please review the errors above.")